### 1. Project Setup — GitHub Repository

Clone the project repository from GitHub and install the required Python dependencies to prepare the Google Colab environment for dataset processing, CNN training and evaluation.

In [ ]:

!git clone https://github.com/antonypradeep54/Deep-Learning---Visual-Pattern-Recognition---Image-Classification.git
%cd Deep-Learning---Visual-Pattern-Recognition---Image-Classification
!pip install -r requirements.txt


### 1.1. Record the Execution Environment

Print the actual Python and library versions used in this Colab runtime. Save the output when preparing a reproducible GitHub release. The dependency file specifies compatible ranges until these versions are confirmed and tested.


In [ ]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version.split()[0])
for package in ["tensorflow", "numpy", "pandas", "scikit-learn", "matplotlib", "Pillow", "pytest"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")


### 2. Connect Google Drive and Configure Project Directory

Mount Google Drive before checking for the dataset. The project folder can be customized with `CNN_PROJECT_DIR`. The expected input is the Kaggle Dogs vs Cats `train.zip` archive; images and checkpoints are not distributed with this notebook.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Drive before accessing files in MyDrive.
drive.mount("/content/drive")

# Override this environment variable if your project is stored elsewhere.
PROJECT_DIR = Path(os.environ.get(
    "CNN_PROJECT_DIR",
    "/content/drive/MyDrive/AI-PM-CT/"
    "Deep-Learning--Visual-Pattern-Recognition--Image-Classification"
))
DATASET_ZIP = PROJECT_DIR / "train.zip"
print("Project directory:", PROJECT_DIR)
print("Dataset exists:", DATASET_ZIP.is_file())
if DATASET_ZIP.is_file():
    print("Dataset size (MB):", round(DATASET_ZIP.stat().st_size / 1024**2, 2))

### 3. Locate and Verify the Dataset

Locate the train.zip archive in Google Drive and verify its file size and path before extraction. This confirms that the dataset is accessible to the Colab environment.

In [ ]:
# Use the configured dataset location; do not search all of MyDrive.
zip_path = DATASET_ZIP
if not zip_path.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found: {zip_path}. "
        "Upload train.zip to PROJECT_DIR or set CNN_PROJECT_DIR."
    )
print("Dataset found:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

### 4. Extract the Image Dataset

Extract the Cats vs Dogs image dataset from the ZIP archive into Colab's local storage for faster processing. Verify the number of extracted images before proceeding with data validation and splitting.

In [ ]:
from pathlib import Path
import zipfile

extract_path = Path("/content/data/raw/pets")
extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    print("Total files in ZIP:", len(zip_ref.namelist()))
    zip_ref.extractall(extract_path)

print("Extraction completed!")

images = list(extract_path.rglob("*.jpg"))
print("Total JPG images extracted:", len(images))
print("Extraction location:", extract_path)

### 5. Dataset Validation and Stratified Splitting

Validate the extracted Cats vs Dogs images, identify corrupted files and exact duplicates, and verify class labels. Generate reproducible training (70%), validation (15%) and testing (15%) splits while preserving the class distribution and keeping the test dataset independent for final model evaluation.

In [ ]:

import subprocess
from pathlib import Path

repo_path = Path(
    "/content/Deep-Learning---Visual-Pattern-Recognition---Image-Classification"
)

result = subprocess.run(
    [
        "python",
        "scripts/prepare_dataset.py",
        "--input", "/content/data/raw/pets",
        "--output", "/content/data/processed",
        "--seed", "42"
    ],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("Error:", result.stderr)
    raise RuntimeError("Dataset preparation failed")


### 6. Review Rejected Images

Inspect images excluded during dataset preparation and summarize the reasons for rejection, such as corrupted files, duplicate images or conflicting labels. This ensures data quality decisions are transparent and documented before model training.

In [ ]:

import pandas as pd

rejected_path = "/content/data/processed/rejected.csv"

rejected_df = pd.read_csv(rejected_path)

print("Total rejected images:", len(rejected_df))

print("\nRejection reasons:")
print(rejected_df["reason"].value_counts())

print("\nRejected image details:")
display(rejected_df)


### 7. Preserve Dataset Validation Manifests

Save the validated dataset split manifest (splits.csv) and rejected-image manifest (rejected.csv) to Google Drive.

These files preserve the reproducible train, validation and test splits, along with records of rejected duplicate images, preventing the need to repeat dataset validation after a Colab runtime restart.

In [ ]:

from pathlib import Path
import shutil

source = Path("/content/data/processed")

destination = PROJECT_DIR / "manifests"

destination.mkdir(parents=True, exist_ok=True)

for filename in ["splits.csv", "rejected.csv"]:
    source_file = source / filename
    destination_file = destination / filename

    if not source_file.exists():
        raise FileNotFoundError(
            f"Missing source manifest: {source_file}"
        )

    shutil.copy2(source_file, destination_file)

    print(f"Saved: {destination_file.name}")
    print(f"Size: {destination_file.stat().st_size:,} bytes")

print("\nDataset manifests successfully preserved in Google Drive.")


### 8. Verify TensorFlow and GPU Availability

Verify the installed TensorFlow version and identify available GPU devices. This confirms whether the Colab environment is ready for accelerated CNN training and helps document the project's execution environment.

In [ ]:

import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

print("GPU devices detected:", len(gpus))

for gpu in gpus:
    print(gpu)


### 9. Verify Project Environment After GPU Activation

Check the availability of the GitHub repository, Google Drive dataset archive, extracted images, and dataset validation manifests after switching to the GPU runtime. Identify any missing resources that must be restored before continuing with CNN preprocessing and training.

In [ ]:

from pathlib import Path

repo = Path(
    "/content/Deep-Learning---Visual-Pattern-Recognition---Image-Classification"
)

drive_folder = PROJECT_DIR

zip_file = drive_folder / "train.zip"
images_folder = Path("/content/data/raw/pets")
splits_file = Path("/content/data/processed/splits.csv")
rejected_file = Path("/content/data/processed/rejected.csv")

print("GitHub repository:", repo.exists())
print("Google Drive mounted:", Path("/content/drive/MyDrive").exists())
print("train.zip accessible:", zip_file.exists())
print("Extracted images folder:", images_folder.exists())
print("Dataset split manifest:", splits_file.exists())
print("Rejected images manifest:", rejected_file.exists())


### 10. Image Preprocessing and TensorFlow Data Pipeline

Prepare the validated Cats vs Dogs dataset for CNN training by resizing images to 128 × 128 pixels, normalizing pixel values to the range 0–1, and converting the images into TensorFlow datasets.

Use a batch size of 32, shuffle the training dataset, and preserve separate validation and testing datasets to ensure reliable model evaluation.

In [ ]:

import pandas as pd
import tensorflow as tf
from pathlib import Path

# Configuration
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

DATA_ROOT = Path("/content/data/raw/pets")
MANIFEST = Path("/content/data/processed/splits.csv")

# Load validated dataset manifest
df = pd.read_csv(MANIFEST)

print("Total validated images:", len(df))
print("\nDataset distribution:")
print(df.groupby(["split", "label"]).size())

# Image preprocessing and TensorFlow dataset creation
def create_dataset(split, shuffle=False):
    subset = df[df["split"] == split].copy()

    paths = [
        str(DATA_ROOT / relative_path)
        for relative_path in subset["path"]
    ]

    labels = (subset["label"] == "Dog").astype("float32").tolist()

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    def preprocess(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(
            image,
            channels=3,
            expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    dataset = dataset.map(
        preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Create training, validation and testing datasets
train_ds = create_dataset("train", shuffle=True)
val_ds = create_dataset("val")
test_ds = create_dataset("test")

print("\nTensorFlow datasets created successfully.")

# Verify one batch
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print(
        "Pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )


### 11. Training Data Augmentation

Apply random horizontal flipping, rotation and zoom to training images to improve CNN generalization and reduce overfitting.

Augmentation is applied only during model training. Validation and testing images remain unchanged to ensure reliable performance evaluation.

In [ ]:

import tensorflow as tf

# Training-only image augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(
        "horizontal",
        seed=42
    ),
    tf.keras.layers.RandomRotation(
        0.1,
        seed=42
    ),
    tf.keras.layers.RandomZoom(
        0.1,
        seed=42
    )
], name="data_augmentation")

# Verify augmentation output
for images, labels in train_ds.take(1):
    augmented_images = data_augmentation(
        images,
        training=True
    )

    print("Original batch shape:", images.shape)
    print("Augmented batch shape:", augmented_images.shape)

    print(
        "Augmented pixel range:",
        float(tf.reduce_min(augmented_images)),
        "to",
        float(tf.reduce_max(augmented_images))
    )


### 12. CNN Model Architecture

Build a Convolutional Neural Network (CNN) to classify images as cats or dogs.

The model uses convolutional layers to learn visual features, max-pooling layers to reduce spatial dimensions, and dense layers to perform binary classification.

Training-only data augmentation is integrated into the model to improve generalization. The final sigmoid layer produces the predicted probability that an image contains a dog.

In [ ]:

import tensorflow as tf

tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),

    # Apply augmentation only during training
    data_augmentation,

    # First convolutional block
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Second convolutional block
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Third convolutional block
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Classification layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    # Binary classification: Cat = 0, Dog = 1
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="cats_vs_dogs_cnn")

model.summary()


### 13. CNN Model Compilation

Compile the CNN using the Adam optimizer with a learning rate of 0.001 and Binary Cross-Entropy as the loss function.

Track accuracy, precision, recall, and ROC-AUC to evaluate classification performance beyond accuracy alone.

The compiled model will be ready for training on the validated Cats vs Dogs dataset.

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("CNN model compiled successfully.")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Loss: Binary Cross-Entropy")
print("Metrics: Accuracy, Precision, Recall, ROC-AUC")


### 14. CNN Model Training

Train the compiled CNN using the validated training dataset and monitor its performance against the independent validation dataset.

Use early stopping to reduce overfitting and restore the best model weights. Save the best-performing model checkpoint to Google Drive so it remains available after the Colab session ends.

Reserve the test dataset for final evaluation after model development is complete.

In [ ]:

from pathlib import Path
import tensorflow as tf

# Persistent checkpoint location in Google Drive
checkpoint_dir = PROJECT_DIR / "checkpoints"

checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "best_cnn_model.keras"

# Training callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# Train the CNN
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed.")
print("Best model checkpoint:", checkpoint_path)
print("Checkpoint exists:", checkpoint_path.exists())


### 15. Training Performance Visualization

Visualize training and validation accuracy and loss across epochs to assess CNN learning behavior, convergence, and potential overfitting.

Identify the epoch with the lowest validation loss and confirm the effectiveness of early stopping before evaluating the model on the independent test dataset.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

epochs = range(1, len(history.history["loss"]) + 1)

best_epoch = np.argmin(history.history["val_loss"]) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(
    epochs, history.history["accuracy"],
    label="Training Accuracy"
)
axes[0].plot(
    epochs, history.history["val_accuracy"],
    label="Validation Accuracy"
)
axes[0].axvline(
    best_epoch, linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)
axes[0].set_title("CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(
    epochs, history.history["loss"],
    label="Training Loss"
)
axes[1].plot(
    epochs, history.history["val_loss"],
    label="Validation Loss"
)
axes[1].axvline(
    best_epoch, linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)
axes[1].set_title("CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    round(min(history.history["val_loss"]), 4)
)


### 16. Independent Test Evaluation

Evaluate the best CNN model on the independent test dataset, which was not used for training or model selection.

Calculate accuracy, precision, recall, F1-score, ROC-AUC and the confusion matrix to assess overall classification performance and identify the types of prediction errors.

Use Cat as class 0 and Dog as class 1, with a classification threshold of 0.5.

In [ ]:

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Load the best saved model
best_model = tf.keras.models.load_model(
    str(checkpoint_path)
)

# Collect true labels and predicted probabilities
y_true = []
y_prob = []

for images, labels in test_ds:
    probabilities = best_model(
        images, training=False
    ).numpy().reshape(-1)

    y_true.extend(labels.numpy())
    y_prob.extend(probabilities)

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)

# Convert probabilities to predicted classes
y_pred = (y_prob >= 0.5).astype(int)

# Calculate evaluation metrics
print("Independent Test Evaluation")
print("---------------------------")
print("Test images:", len(y_true))
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred), 4))
print("Recall:", round(recall_score(y_true, y_pred), 4))
print("F1-score:", round(f1_score(y_true, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Cat", "Dog"]
    )
)

# Display confusion matrix
cm = confusion_matrix(y_true, y_pred)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Cat", "Dog"]
)

display.plot(cmap="Blues", values_format="d")
plt.title("CNN — Independent Test Confusion Matrix")
plt.show()


### 17. Verify Trained Model Checkpoint

Verify that the best-performing CNN model checkpoint has been successfully saved to Google Drive. Confirm the file exists and check its size to ensure the trained model is preserved for future evaluation and inference without retraining.

In [ ]:

from pathlib import Path

checkpoint = PROJECT_DIR / "checkpoints" / "best_cnn_model.keras"

print("Checkpoint exists:", checkpoint.is_file())

if checkpoint.is_file():
    print(
        "Checkpoint size (MB):",
        round(checkpoint.stat().st_size / 1024**2, 2)
    )


### 18. CNN Classification Error Analysis

Analyze the CNN's incorrect predictions using the independent test results.

Examine false positives (cats incorrectly predicted as dogs) and false negatives (dogs incorrectly predicted as cats). Visualize representative misclassified images and their predicted probabilities to understand model limitations and identify potential improvements.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Use predictions from independent test evaluation (Step 16)
cm = confusion_matrix(y_true, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print("\nError Breakdown")
print("----------------")
print("Correctly classified cats:", tn)
print("Cats predicted as dogs (FP):", fp)
print("Dogs predicted as cats (FN):", fn)
print("Correctly classified dogs:", tp)
print("Total incorrect predictions:", fp + fn)

# Identify incorrect predictions
false_positive_indices = np.where(
    (y_true == 0) & (y_pred == 1)
)[0]

false_negative_indices = np.where(
    (y_true == 1) & (y_pred == 0)
)[0]

print("\nFalse positives:", len(false_positive_indices))
print("False negatives:", len(false_negative_indices))


### 19. Visualize Misclassified Images

Display representative false-positive and false-negative predictions from the independent test dataset.

Compare the actual class, predicted class and model confidence to investigate potential causes of classification errors and identify opportunities for improving CNN generalization.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Select three examples from each error category
fp_examples = false_positive_indices[:3]
fn_examples = false_negative_indices[:3]

selected_indices = np.concatenate([
    fp_examples,
    fn_examples
])

# Load test images in the same order as predictions
test_images = np.concatenate([
    images.numpy()
    for images, labels in test_ds
], axis=0)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, idx in zip(axes.flat, selected_indices):
    actual = "Dog" if y_true[idx] == 1 else "Cat"
    predicted = "Dog" if y_pred[idx] == 1 else "Cat"

    confidence = (
        y_prob[idx]
        if y_pred[idx] == 1
        else 1 - y_prob[idx]
    )

    ax.imshow(test_images[idx])
    ax.set_title(
        f"Actual: {actual}\n"
        f"Predicted: {predicted}\n"
        f"Confidence: {confidence:.1%}"
    )
    ax.axis("off")

plt.suptitle("CNN Misclassified Images", fontsize=14)
plt.tight_layout()
plt.show()


### 20. Save Baseline Model Evaluation Results

Preserve the CNN baseline experiment results in Google Drive to support reproducibility, model comparison and GitHub documentation.

Save the training history, independent test evaluation metrics, confusion matrix, model configuration and training parameters. These artifacts establish a documented baseline for evaluating future model improvements.

Future model selection will use validation performance, with the independent test results retained as the baseline's final evaluation.

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Google Drive experiment results folder
results_dir = PROJECT_DIR / "results" / "baseline_cnn"

results_dir.mkdir(parents=True, exist_ok=True)

# 1. Save training history
history_df = pd.DataFrame(history.history)
history_df.index = history_df.index + 1
history_df.index.name = "epoch"

history_df.to_csv(
    results_dir / "training_history.csv"
)

# 2. Calculate and save test metrics
metrics = {
    "model": "Custom CNN",
    "test_images": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred)),
    "recall": float(recall_score(y_true, y_pred)),
    "f1_score": float(f1_score(y_true, y_pred)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "classification_threshold": 0.5,
    "positive_class": "Dog",
    "best_epoch": int(
        np.argmin(history.history["val_loss"]) + 1
    ),
    "best_validation_loss": float(
        min(history.history["val_loss"])
    )
}

with open(results_dir / "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# 3. Save confusion matrix
cm = confusion_matrix(y_true, y_pred)

pd.DataFrame(
    cm,
    index=["Actual_Cat", "Actual_Dog"],
    columns=["Predicted_Cat", "Predicted_Dog"]
).to_csv(results_dir / "confusion_matrix.csv")

# 4. Save model architecture
with open(results_dir / "model_architecture.json", "w") as f:
    f.write(best_model.to_json())

# 5. Save experiment configuration
configuration = {
    "image_size": [128, 128],
    "batch_size": 32,
    "random_seed": 42,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss_function": "BinaryCrossentropy",
    "max_epochs": 15,
    "early_stopping_patience": 3,
    "model_checkpoint": "best_cnn_model.keras",
    "training_images": 17494,
    "validation_images": 3748,
    "test_images": 3752,
    "data_augmentation": [
        "RandomFlip(horizontal)",
        "RandomRotation(0.1)",
        "RandomZoom(0.1)"
    ]
}

with open(results_dir / "experiment_config.json", "w") as f:
    json.dump(configuration, f, indent=4)

# Verify saved files
print("Baseline CNN artifacts saved successfully.\n")

for file in sorted(results_dir.iterdir()):
    print(file.name, "-", file.stat().st_size, "bytes")

print("\nResults directory:", results_dir)


### 21. Interactive Image Classification Using the Trained CNN

Demonstrate how the trained CNN can classify a new image uploaded by a user.

The application loads the saved CNN checkpoint, preprocesses the uploaded image using the same input dimensions and normalization applied during training, and predicts whether the image contains a cat or dog.

The predicted class and model probability are displayed alongside the uploaded image.

**Note:** The model is designed for binary classification between cats and dogs. Its prediction score is not a guarantee of correctness, and images outside these two categories may still receive a cat or dog prediction.

In [ ]:

from google.colab import files
from pathlib import Path
from PIL import Image
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import io

# Load the saved CNN checkpoint
checkpoint_path = PROJECT_DIR / "checkpoints" / "best_cnn_model.keras"

cnn_model = tf.keras.models.load_model(checkpoint_path)

print("CNN model loaded successfully.")
print("Please upload a cat or dog image.\n")

# Upload an image from your computer
uploaded = files.upload()

for filename, file_bytes in uploaded.items():

    # Open and convert image to RGB
    image = Image.open(
        io.BytesIO(file_bytes)
    ).convert("RGB")

    # Resize and normalize, matching CNN training
    # Match training: tf.image.resize uses bilinear interpolation.
    resized_image = tf.image.resize(
        np.asarray(image, dtype=np.float32), (128, 128),
        method="bilinear"
    ).numpy()

    image_array = np.asarray(
        resized_image,
        dtype=np.float32
    ) / 255.0

    # Add batch dimension
    image_batch = np.expand_dims(
        image_array,
        axis=0
    )

    # Generate prediction
    dog_probability = float(
        cnn_model.predict(image_batch, verbose=0)[0][0]
    )

    cat_probability = 1.0 - dog_probability

    predicted_class = (
        "Dog" if dog_probability >= 0.5 else "Cat"
    )

    predicted_score = max(
        dog_probability,
        cat_probability
    )

    # Display result
    plt.figure(figsize=(6, 5))
    plt.imshow(image)
    plt.axis("off")

    plt.title(
        f"Prediction: {predicted_class}\n"
        f"Prediction score: {predicted_score:.2%}"
    )

    plt.show()

    print(f"Image: {filename}")
    print(f"Predicted class: {predicted_class}")
    print(f"Cat probability: {cat_probability:.2%}")
    print(f"Dog probability: {dog_probability:.2%}")
